In [ ]:
import numpy as np
import json
import pandas as pd
from datetime import timedelta
import pycountry
from numpyro import distributions as dist
import arviz as az
from IPython.display import display, Markdown

from emu_renewal.constants import DATA_PATH, RERUNS, FULL_RUN, TIMEOUTS, DATE_FORMAT, RUN_DATA_DELAY
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.priors import get_standard_priors
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget
from emu_renewal.plotting import plot_input_recovery
from emu_renewal.utils import get_analysis_paths

In [ ]:
mob_source = "fb_singletile_mob"
country = "France"
iso3 = pycountry.countries.lookup(country).alpha_3
name = pycountry.countries.lookup(country).name
continent = get_cont_of_country(iso3)
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
display(Markdown(f"mobility approach: {mob_source}"))
display(Markdown(f"\n country: {name}"))

In [ ]:
# Select random set of scalar parameters from calibration
path = analysis_paths[iso3][mob_source]
idata = az.from_netcdf(path / "idata_filtered.nc")
post = idata.posterior
chain = np.random.randint(post.sizes["chain"])
draw = np.random.randint(post.sizes["draw"])
draw_params = post.isel(chain=chain, draw=draw)
scalar_params = {p: float(v) for p, v in draw_params.items() if v.ndim == 0}
multi_params = {p: v.values for p, v in draw_params.items() if v.ndim > 0 and p != "proc"}

In [ ]:
# Build the model
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
var_names = ["eu"] + alpha_var  # Not applicable for Omicron-era countries
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, alpha_seed, mob_provider, True, False)
thinning = 7
times = model.epoch.number_to_datetime(pd.Series(model.model_times))[::thinning]

priors = get_standard_priors(len(var_names), "weekly_admissions", iso3, continent, False)
prior_means = {k: v if isinstance(v, float) else v.mean for k, v in priors.items()}

In [ ]:
# Get synthetic indicators for calibration
identify_params = {"mob_exp": 1.0}
proc = np.random.normal(0.0, 0.1, model.x_proc_vals.shape[0])
params = scalar_params | multi_params | identify_params | prior_means
results = model.renewal_func(**params | {"proc": proc})
indicators = ["weekly_deaths", "weekly_cases"]
outputs = {i: pd.Series(results[i][::thinning], index=times) for i in indicators}
targets = {ind: SharedDispTarget(targ, weight=20) for ind, targ in outputs.items()}
calib, mcmc = run_calibration(model, params, targets, True, 1000)
display(plot_input_recovery(priors, calib, az.from_numpyro(mcmc), outputs, identify_params, proc))